
# Gibbs Free Energy of Formation **298 K Only** Estimator

Based on the methodology described in the manuscript and SI, we **do not** use the QC‑derived thermochemistry to build \(G(T)\) at ALL temperatures. Artifacts in the frequency/solvation treatment make entropies unreliable away from room temperature, so we restrict the quantum‑chemistry estimator to a **single point: 298 K**. We then use our separate analog‑slope approach (see manuscript/SI) to extend to other temperatures for modeling. This notebook therefore calculates **ΔG\_f(298 K) only** for a target molecule.

> This implementation automates the well‑known **Ochterski 3‑step method** at 298 K. Please cite: **Ochterski, J. W. (2000). _Thermochemistry in Gaussian_. Gaussian Inc., 1(1).**

---

## What this notebook expects

- A CSV called **`thermo_results.csv`** with one row per Gaussian output file and the following columns (case‑sensitive):
  - `filename` (e.g., `adenine_PCM_298K.log`, `Carbon_PCM_298K.log`, `H2_PCM_298K.log`)
  - `electronic_energy` (Hartree)
  - `ZPE` (Hartree)
  - `dH` (Hartree; thermal enthalpy correction **including** ZPE)
  - `entropy` (cal/mol·K)

- Required filenames at **298 K** only:
  - Molecule: `{molecule}_PCM_298K.log`
  - Atomic references used per atom: `H2_PCM_298K.log`, `O2_PCM_298K.log`, `N2_PCM_298K.log`, and a carbon proxy (see below).

---

## How to use

1. Put `thermo_results.csv` next to this notebook.
2. Run the code cell below. You’ll be prompted for:
   - **Molecule name prefix** (e.g., `adenine`) which matches `adenine_PCM_298K.log` in your CSV.
   - **Stoichiometry** as `C:5,H:5,N:5`.
3. The script prints **ΔG\_f(298 K)** in J/mol.

**Output.**  
A one‑row pandas DataFrame with:
- `T (K)` = 298
- `Gf (J/mol)` = ΔG\_f(298 K)

---

## Method summary (298 K only)

- Compute ΔH\_f(0 K) from atomic ΔH\_f(0 K), isolated‑atom SPEs, and the molecule’s ZPE.
- Add thermal enthalpy corrections at 298 K for the molecule and for atoms (H₂/O₂/N₂ files are divided by 2 to get **per‑atom** values).
- Subtract TS at 298 K.
- Return ΔG\_f(298 K) in J/mol.

**Caveats.**
- This notebook **only** evaluates 298 K. For \(G(T)\) outside 298 K, see the manuscript/SI analog‑slope procedure.
- Carbon atom entropy uses a small **proxy term** (BASE\_S\_C) at 298 K (no T‑trend here). If you later generate a better carbon reference, update that constant.
- All numbers (SPEs, tabulated ΔH\_f(0 K)) are project‑specific; adjust if your level of theory or references differ.
- All quantum chemical calculations were performed using the B3LYP/6-311++G(2df,2p) to ensure consistency across molecular optimizations, frequency analyses, and thermochemical estimations.

**Reference.**
- Ochterski, J. W. (2000). _Thermochemistry in Gaussian_. Gaussian Inc., 1(1).


In [1]:

# ΔG_f(298 K) Estimator (Ochterski-based); 298 K ONLY
# ----------------------------------------------------
# This script reads quantum-chemistry thermo outputs from `thermo_results.csv`
# and computes ΔG_f at *298 K only* for a target molecule.
#
# Why 298 K only?
# The manuscript/SI describe entropy/solvation artifacts away from 298 K in our QC setup,
# so we use QC only at 298 K and use a separate analog-slope method for other T.
#
# Reference: Ochterski, J.W. (2000) Thermochemistry in Gaussian, Gaussian Inc.
#
# ----------------------------------------------------

import os
import pandas as pd

# ---------- Constants & required column names ----------
HARTREE_TO_KCAL = 627.509
CAL_TO_KCAL     = 1e-3
KCAL_TO_KJ      = 4.184

COL_FILENAME      = "filename"
COL_ELECTRONIC    = "electronic_energy"
COL_ZPE           = "ZPE"
COL_H_CORR        = "dH"
COL_ENTROPY       = "entropy"

# Carbon entropy proxy at 298 K (kcal/mol-K). See manuscript/SI caveat.
BASE_S_C = 0.001371892925

# Mapping elements to the (diatomic) filename prefixes at 298 K
ELEMENT_PREFIX = {
    "C": "Carbon",  # entropy handled by BASE_S_C; enthalpy corr used if present
    "H": "H2",
    "O": "O2",
    "N": "N2",
}

def load_data(csv_path: str) -> pd.DataFrame:
    """Load thermo_results.csv as a DataFrame."""
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"{csv_path} not found")
    return pd.read_csv(csv_path)

def get_user_inputs():
    """
    Prompt for the target molecule and stoichiometry.
    Example:
      molecule: adenine
      stoich:   C:5,H:5,N:5
    """
    molecule = input("Enter molecule name prefix (e.g. 'adenine'): ").strip()
    stoich_input = input("Enter stoichiometry as comma-separated [el:count] (e.g. C:5,H:5,N:5): ")
    stoich = {}
    for pair in stoich_input.split(","):
        el, cnt = pair.split(":")
        stoich[el.strip()] = int(cnt)
    return molecule, stoich

def compute_zero_kelvin_refs(stoich: dict[str, int]):
    """
    0 K references for Ochterski:
      - E_atoms_h: sum of isolated-atom SPEs (Hartree)
      - sum_hf0_h: sum of atomic ΔH_f(0 K) (Hartree)
    """
    ATOMIC_SPE = {"C": -37.858358, "H": -0.50228348, "O": -75.091525, "N": -54.600727}  # Hartree (project values)
    HF_REF     = {"H": 51.63, "O": 58.99, "C": 169.98, "N": 112.53}                      # kcal/mol (project values)

    E_atoms_h = sum(stoich[el] * ATOMIC_SPE[el] for el in stoich)
    sum_hf0_kcal = sum(stoich[el] * HF_REF[el] for el in stoich)
    sum_hf0_h = sum_hf0_kcal / HARTREE_TO_KCAL
    return E_atoms_h, sum_hf0_h

def get_row_exact(df: pd.DataFrame, fname: str) -> pd.Series:
    """Return the exact-matching row for `filename == fname` (raises if missing)."""
    row = df.loc[df[COL_FILENAME] == fname]
    if row.empty:
        raise ValueError(f"Required entry not found in CSV: {fname}")
    return row.iloc[0]

def deltaH_T_minus_ZPE(row: pd.Series) -> float:
    """Return ΔH(T) - ZPE (in Hartree) for the provided row (T=298 K here)."""
    return row[COL_H_CORR] - row[COL_ZPE]

def compute_DGf_298K(df: pd.DataFrame, molecule: str, stoich: dict[str, int]) -> pd.DataFrame:
    """
    Compute ΔG_f at 298 K only.
    Returns a 1-row DataFrame: T (K)=298, Gf (J/mol).
    """
    T = 298
    # Fetch molecule row at 298 K
    mol_fname = f"{molecule}_PCM_{T}K.log"
    mrow = get_row_exact(df, mol_fname)

    # (1) Atomization energy at 0 K: E_atoms - (E_mol + ZPE_mol)
    E_atoms_h, sum_hf0_h = compute_zero_kelvin_refs(stoich)
    E_mol_h   = mrow[COL_ELECTRONIC]
    ZPE_mol_h = mrow[COL_ZPE]
    atom_h    = E_atoms_h - (E_mol_h + ZPE_mol_h)

    # (2) ΔH_f(0 K)
    Hf0_mol_h = sum_hf0_h - atom_h

    # (3) Thermal enthalpy corrections at 298 K
    # Molecular correction (beyond ZPE)
    dH_mol_h = deltaH_T_minus_ZPE(mrow)

    # Atomic corrections: diatomics per atom; carbon uses Carbon_PCM_298K.log if present
    sum_dH_atoms_h = 0.0
    for el, n in stoich.items():
        prefix = ELEMENT_PREFIX[el]
        afname = f"{prefix}_PCM_{T}K.log"
        arow = get_row_exact(df, afname)
        dH_atomish = deltaH_T_minus_ZPE(arow)
        if el in ("H", "O", "N"):
            dH_atomish /= 2.0  # per atom from diatomic per-molecule value
        # For C, if Carbon file is present we use its enthalpy correction directly.
        sum_dH_atoms_h += n * dH_atomish

    # ΔH_f(298 K): convert Hartree → kcal/mol
    Hf_T_h    = Hf0_mol_h + dH_mol_h - sum_dH_atoms_h
    Hf_T_kcal = Hf_T_h * HARTREE_TO_KCAL

    # (4) Entropy term at 298 K
    S_mol_cal = mrow[COL_ENTROPY]
    sum_S_atoms_cal = 0.0
    for el, n in stoich.items():
        if el in ("H", "O", "N"):
            prefix = ELEMENT_PREFIX[el]
            afname = f"{prefix}_PCM_{T}K.log"
            arow = get_row_exact(df, afname)
            S_diat_cal = arow[COL_ENTROPY]
            sum_S_atoms_cal += n * (S_diat_cal / 2.0)  # per atom
        else:
            # Carbon proxy entropy at 298 K (no temperature ramp here)
            sum_S_atoms_cal += n * (BASE_S_C * 1000.0)

    dS_cal  = S_mol_cal - sum_S_atoms_cal         # cal/mol-K
    dS_kcal = dS_cal * CAL_TO_KCAL                # kcal/mol-K

    # (5) ΔG_f(298 K) = ΔH_f(298 K) - T * ΔS(298 K); convert kcal/mol → J/mol
    Gf_kcal = Hf_T_kcal - T * dS_kcal
    Gf_J    = Gf_kcal * KCAL_TO_KJ * 1000.0

    return pd.DataFrame([{"T (K)": T, "Gf (J/mol)": Gf_J}])

def main():
    csv_path = "thermo_results.csv"
    df = load_data(csv_path)
    molecule, stoich = get_user_inputs()
    out = compute_DGf_298K(df, molecule, stoich)
    print("Molecule in question:", molecule)
    print(out)

if __name__ == "__main__":
    main()


Enter molecule name prefix (e.g. 'adenine'):  beta-alanine
Enter stoichiometry as comma-separated [el:count] (e.g. C:5,H:5,N:5):  C:3, H:7, N:1, O:2


Molecule in question: beta-alanine
   T (K)     Gf (J/mol)
0    298 -290588.773538
